# Datasets

In [5]:
from datasets import load_dataset

dataset = load_dataset("rotten_tomatoes")

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [18]:
dataset['validation'][0]['text']

'compassionately explores the seemingly irreconcilable situation between conservative christian parents and their estranged gay and lesbian children .'

In [19]:
dataset['validation']['text'][0]

'compassionately explores the seemingly irreconcilable situation between conservative christian parents and their estranged gay and lesbian children .'

In [25]:
from time import time

start = time()
dataset['validation'][0]['text']
end = time()
print(f"Elapsed time: {end-start:.4f}s")

start = time()
dataset['validation']['text'][0]
end = time()
print(f"Elapsed time: {end-start:.4f}s")

Elapsed time: 0.0010s
Elapsed time: 0.0020s


In [26]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [32]:
tokenizer(dataset['validation']['text'][0])

{'input_ids': [101, 29353, 2135, 15102, 1996, 9428, 20868, 2890, 8663, 6895, 20470, 2571, 3663, 2090, 4603, 3017, 3008, 1998, 2037, 24211, 5637, 1998, 11690, 2336, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [40]:
def tokenization(x):
    return tokenizer(x['text'])

dataset.map(tokenization, batch_size=3000)

Map: 100%|████████████████████████████████████████████████████████████████| 1066/1066 [00:00<00:00, 4593.90 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
})

In [41]:
from datasets import Dataset

ds = Dataset.from_dict({
    "pokemon": ['squirtle', 'bulbasaur'],
    "type": ['grass', 'water']
})

ds

Dataset({
    features: ['pokemon', 'type'],
    num_rows: 2
})

# Fine Tuning, Ejemplo (Clasificación)

In [44]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/main/amazon_sports.csv")
df.head()

,stars,review_body,review_title,product_category
0,1,Nunca llego el pedido y el vendedor pasa de to...,No llego nunca,sports
1,1,"No sé como es, porque debería haber llegado ay...",Todavía no ha llegado,sports
2,1,"Guantes cómodos, no lo niego, pero de mala cal...",Guantes de baja calidad,sports
3,1,Hasta hoy no he visto el producto. El pedido h...,Muy Mala experiencia,sports
4,1,"No puedo valorarla porque, después de casi una...",Paquete perdido?,sports


In [45]:
df = df[df.stars.isin([1,5])]
df['good_product'] = (df.stars>3).astype(int)
df.groupby('good_product').size()

good_product
0    2438
1    2512
dtype: int64

In [64]:
ds = Dataset.from_pandas(df)
ds = ds.remove_columns(['stars', 'review_title', 'product_category',  '__index_level_0__'])
ds = ds.train_test_split(test_size=0.2)
ds = ds.rename_column('good_product', 'labels')
ds

DatasetDict({
    train: Dataset({
        features: ['review_body', 'labels'],
        num_rows: 3960
    })
    test: Dataset({
        features: ['review_body', 'labels'],
        num_rows: 990
    })
})

In [65]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)
model.__class__

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification

In [66]:
def tokenize_function(x):
    return tokenizer(x['review_body'], truncation=True)

tokenized_ds = ds.map(tokenize_function, batched=True)

Map: 100%|█████████████████████████████████████████████████████████████████| 990/990 [00:00<00:00, 18436.01 examples/s]


In [67]:
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['review_body', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 3960
    })
    test: Dataset({
        features: ['review_body', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 990
    })
})

# Training

In [68]:
from transformers import TrainingArguments

train_args = TrainingArguments(model_id)

In [69]:
train_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=3,
)

In [75]:
from transformers import Trainer
from transformers import DataCollatorWithPadding

dc = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model, 
    train_args, 
    train_dataset=tokenized_ds['train'],
    tokenizer=tokenizer,
    data_collator=dc,
)

C:\Users\efclprd\AppData\Local\Temp\ipykernel_14636\1091712897.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [72]:
%%time

trainer.train()

Step,Training Loss
500,0.653700
1000,0.499500
1500,0.394000
2000,0.310200
2500,0.285500
3000,0.195100
3500,0.170400


CPU times: total: 4min 10s
Wall time: 4min 14s


TrainOutput(global_step=3960, training_loss=0.33962768593219794, metrics={'train_runtime': 253.4088, 'train_samples_per_second': 46.881, 'train_steps_per_second': 15.627, 'total_flos': 269875676046636.0, 'train_loss': 0.33962768593219794, 'epoch': 3.0})

# Evaluation

In [79]:
predictions = trainer.predict(tokenized_ds['test'])
print(predictions.predictions.shape, predictions.label_ids.shape)

(990, 2) (990,)


In [84]:
import numpy as np

preds = predictions.predictions
y_hat = np.argmax(preds, axis=1)

In [87]:
from sklearn.metrics import accuracy_score, f1_score

accuracy_score(y_pred=y_hat, y_true=ds['test']['labels'])
f1_score(y_pred=y_hat, y_true=ds['test']['labels'])

0.9307317073170732